In [ ]:
# ============================================
# NOTEBOOK 4: BiLSTM + Attention cho Dynamic Recognition
# ============================================
# Xử lý các ký tự động (J, Z) và có thể mở rộng cho tất cả

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

# Dọn dẹp memory
torch.cuda.empty_cache()
gc.collect()
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Cấu hình
BONE_DATA_DIR = "/kaggle/input/datasets/tonirighthere/bone-dgram/bone_data"
OUTPUT_DIR = "/kaggle/working/bilstm_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/checkpoints", exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU Memory at start: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

print("=" * 50)
print("GIAI ĐOẠN 3.2: BiLSTM + ATTENTION - DYNAMIC RECOGNITION")
print("=" * 50)

# ============================================
# 1. LOAD METADATA
# ============================================
print("\n1. Loading metadata...")

# Tìm đúng đường dẫn
if not os.path.exists(f"{BONE_DATA_DIR}/bone_metadata.json"):
    BONE_DATA_DIR = "/kaggle/input/bone_data"
    
bone_metadata_path = f"{BONE_DATA_DIR}/bone_metadata.json"
if not os.path.exists(bone_metadata_path):
    print("ERROR: Không tìm thấy bone_metadata.json!")
    raise FileNotFoundError("Missing bone_metadata.json")

with open(bone_metadata_path, 'r') as f:
    metadata = json.load(f)

NUM_KEYPOINTS = metadata.get('num_keypoints', 21)  # 21 keypoints
FEATURES_PER_KEYPOINT = metadata.get('features_per_keypoint', 3)  # x, y, z
INPUT_FEATURE_SIZE = NUM_KEYPOINTS * FEATURES_PER_KEYPOINT  # 63 features
NUM_CLASSES = metadata['num_classes']
class_mapping = metadata['class_mapping']  # {'A': 0, 'B': 1, ...}
idx_to_class = metadata.get('idx_to_class', {str(v): k for k, v in class_mapping.items()})

print(f"Number of keypoints: {NUM_KEYPOINTS}")
print(f"Features per keypoint: {FEATURES_PER_KEYPOINT}")
print(f"Input feature size: {INPUT_FEATURE_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Class mapping sample: {dict(list(class_mapping.items())[:5])}")

# ============================================
# 2. LOAD KEYPOINTS DATA
# ============================================
print("\n2. Loading keypoints data...")

keypoints_path = f"{BONE_DATA_DIR}/keypoints_array.npy"
labels_path = f"{BONE_DATA_DIR}/labels_array.npy"

if not os.path.exists(keypoints_path):
    keypoints_path = "/kaggle/input/bone_data/keypoints_array.npy"
    labels_path = "/kaggle/input/bone_data/labels_array.npy"

keypoints_array = np.load(keypoints_path)
labels_array = np.load(labels_path)

print(f"Keypoints array shape: {keypoints_array.shape}")
print(f"Labels array shape: {labels_array.shape}")
print(f"Unique labels: {np.unique(labels_array)}")

# ============================================
# 3. TẠO SEQUENCES CHO VIDEO SIMULATION
# ============================================
print("\n3. Creating sequences for video simulation...")

def create_sequences(keypoints, labels, seq_length=30, stride=15):
    """
    Tạo sequences từ keypoints riêng lẻ (mô phỏng video)
    Mỗi sequence là một chuỗi các frame liên tiếp
    """
    sequences = []
    seq_labels = []
    
    # Nhóm theo class
    unique_labels = np.unique(labels)
    
    for label in unique_labels:
        # Bỏ qua class nothing (ID 27) nếu có
        if label == 27:
            print(f"Skipping class 'nothing' (ID: 27)")
            continue
            
        class_indices = np.where(labels == label)[0]
        class_keypoints = keypoints[class_indices]
        
        # Tạo sequences từ các frame cùng class
        for i in range(0, len(class_keypoints) - seq_length, stride):
            seq = class_keypoints[i:i+seq_length]
            if len(seq) == seq_length:
                sequences.append(seq)
                seq_labels.append(label)
    
    return np.array(sequences), np.array(seq_labels)

# Cấu hình sequence
SEQ_LENGTH = 30  # 30 frames per sequence (khoảng 1 giây ở 30fps)
STRIDE = 15      # Overlap giữa các sequences

print(f"Creating sequences (seq_length={SEQ_LENGTH}, stride={STRIDE})...")
X_seq, y_seq = create_sequences(keypoints_array, labels_array, seq_length=SEQ_LENGTH, stride=STRIDE)
print(f"Sequences shape: {X_seq.shape}")
print(f"Sequence labels shape: {y_seq.shape}")
print(f"Unique labels in sequences: {np.unique(y_seq)}")

if len(X_seq) == 0:
    raise Exception("No sequences created! Check if there are enough samples per class.")

# ============================================
# 4. SPLIT DATA
# ============================================
print("\n4. Splitting data (70/15/15)...")

X_train, X_temp, y_train, y_temp = train_test_split(
    X_seq, y_seq, 
    test_size=0.3, 
    stratify=y_seq,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    stratify=y_temp,
    random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

# Reshape cho BiLSTM input: (batch, seq_len, features)
X_train_flat = X_train.reshape(X_train.shape[0], X_train.shape[1], -1)
X_val_flat = X_val.reshape(X_val.shape[0], X_val.shape[1], -1)
X_test_flat = X_test.reshape(X_test.shape[0], X_test.shape[1], -1)

print(f"Input shape after flatten: {X_train_flat.shape}")

# ============================================
# 5. DATASET CLASS
# ============================================

class KeypointsSequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = KeypointsSequenceDataset(X_train_flat, y_train)
val_dataset = KeypointsSequenceDataset(X_val_flat, y_val)
test_dataset = KeypointsSequenceDataset(X_test_flat, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Batch size: {batch_size}")

# ============================================
# 6. ATTENTION MECHANISMS
# ============================================

class AdditiveAttention(nn.Module):
    """Additive attention (Bahdanau attention)"""
    def __init__(self, hidden_size):
        super(AdditiveAttention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x: (batch, seq_len, hidden_size)
        attention_weights = torch.softmax(self.attention(x).squeeze(-1), dim=1)
        context = torch.sum(x * attention_weights.unsqueeze(-1), dim=1)
        return context, attention_weights

class MultiHeadAttention(nn.Module):
    """Multi-head attention (Transformer style)"""
    def __init__(self, hidden_size, num_heads=4):
        super(MultiHeadAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        assert hidden_size % num_heads == 0, "hidden_size must be divisible by num_heads"
        
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        # Linear projections
        Q = self.query(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = torch.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        context = torch.matmul(attention_weights, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        
        return self.out(context), attention_weights

# ============================================
# 7. BiLSTM + ATTENTION MODEL
# ============================================
print("\n5. Building BiLSTM + Attention model...")

class BiLSTM_Attention(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=29, 
                 dropout=0.3, attention_type='additive'):
        super(BiLSTM_Attention, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.attention_type = attention_type
        
        # Batch Normalization cho input
        self.bn = nn.BatchNorm1d(input_size)
        
        # BiLSTM layers
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention mechanism
        if attention_type == 'additive':
            self.attention = AdditiveAttention(hidden_size * 2)
        else:
            self.attention = MultiHeadAttention(hidden_size * 2, num_heads=4)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size * 2, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        batch_size, seq_len, input_size = x.shape
        
        # Apply batch norm (reshape for BatchNorm1d)
        x_reshaped = x.view(-1, input_size)
        x_norm = self.bn(x_reshaped)
        x = x_norm.view(batch_size, seq_len, input_size)
        
        # BiLSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        
        # Attention
        context, attention_weights = self.attention(lstm_out)
        
        # Fully connected
        out = self.dropout(context)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out, attention_weights

# Model hyperparameters
HIDDEN_SIZE = 128  # Giảm từ 256 xuống 128 để tránh OOM
NUM_LAYERS = 2
DROPOUT = 0.3
ATTENTION_TYPE = 'additive'  # 'additive' hoặc 'multihead'

model = BiLSTM_Attention(
    input_size=INPUT_FEATURE_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
    attention_type=ATTENTION_TYPE
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Attention type: {ATTENTION_TYPE}")
print(f"Hidden size: {HIDDEN_SIZE}")
print(f"Number of LSTM layers: {NUM_LAYERS}")

# Kiểm tra GPU memory
print(f"GPU memory after model: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

# ============================================
# 8. LOSS, OPTIMIZER, SCHEDULER
# ============================================
print("\n6. Setting up training...")

# Class weights cho imbalance
unique_labels_train = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=unique_labels_train, y=y_train)
class_weights_dict = {label: weight for label, weight in zip(unique_labels_train, class_weights)}

# Tạo mảng weights đầy đủ
full_weights = np.ones(NUM_CLASSES, dtype=np.float32)
for label, weight in class_weights_dict.items():
    full_weights[label] = weight

# Class 27 (nothing) có thể không có trong train
if 27 not in unique_labels_train:
    full_weights[27] = 0.0

class_weights_tensor = torch.tensor(full_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ============================================
# 9. TRAINING FUNCTIONS
# ============================================

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for sequences, labels in tqdm(loader, desc="Training"):
        sequences, labels = sequences.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs, _ = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_attention_weights = []
    
    with torch.no_grad():
        for sequences, labels in tqdm(loader, desc="Validation"):
            sequences, labels = sequences.to(device), labels.to(device)
            outputs, attn_weights = model(sequences)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_attention_weights.append(attn_weights.cpu())
    
    return running_loss / len(loader), 100. * correct / total, all_attention_weights

# ============================================
# 10. TRAINING LOOP
# ============================================
print("\n7. Starting training...")

num_epochs = 50
best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 40)
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, _ = validate(model, val_loader, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    scheduler.step(val_loss)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/checkpoints/best_bilstm_model.pth")
        print(f"✓ Saved best model (val_acc: {val_acc:.2f}%)")
    
    # Early stopping
    if val_acc >= 95.0:
        print(f"🎉 Early stopping at epoch {epoch+1}")
        break

# ============================================
# 11. PLOT TRAINING CURVES
# ============================================
print("\n8. Plotting training curves...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_accs, label='Train Accuracy')
axes[1].plot(val_accs, label='Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png")
plt.show()

# ============================================
# 12. EVALUATE ON TEST SET
# ============================================
print("\n9. Evaluating on test set...")

# Load best model
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/checkpoints/best_bilstm_model.pth"))
model.eval()

all_preds = []
all_labels = []
all_attentions = []

with torch.no_grad():
    for sequences, labels in tqdm(test_loader, desc="Testing"):
        sequences = sequences.to(device)
        outputs, attn_weights = model(sequences)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_attentions.append(attn_weights)

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

print(f"\n📊 Test Results:")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")
print(f"F1-Score: {f1*100:.2f}%")

# Classification report
unique_labels_test = np.unique(all_labels)
class_names = [idx_to_class.get(str(i), f"Class_{i}") for i in range(NUM_CLASSES) if i in unique_labels_test]

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix - BiLSTM+Attention')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.show()

# ============================================
# 13. VISUALIZE ATTENTION WEIGHTS
# ============================================
print("\n10. Visualizing attention weights...")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i in range(min(8, len(all_attentions))):
    attn = all_attentions[i]
    if attn.dim() == 3:
        attn = attn.squeeze(0).mean(0)  # Average over heads if multi-head
    elif attn.dim() == 2:
        attn = attn[0]  # Take first batch
    
    attn_np = attn.numpy() if hasattr(attn, 'numpy') else np.array(attn)
    
    axes[i].plot(attn_np)
    axes[i].set_title(f'Sample {i+1} - Attention Weights')
    axes[i].set_xlabel('Time Step')
    axes[i].set_ylabel('Attention Weight')
    axes[i].grid(True)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/attention_weights.png")
plt.show()

# ============================================
# 14. SAVE MODEL
# ============================================
print("\n11. Saving model...")

model_info = {
    'model_type': 'BiLSTM_Attention',
    'attention_type': ATTENTION_TYPE,
    'input_size': INPUT_FEATURE_SIZE,
    'seq_length': SEQ_LENGTH,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'num_classes': NUM_CLASSES,
    'dropout': DROPOUT,
    'classes': [idx_to_class.get(str(i), f"Class_{i}") for i in range(NUM_CLASSES)],
    'test_accuracy': float(accuracy),
    'test_precision': float(precision),
    'test_recall': float(recall),
    'test_f1': float(f1),
    'best_val_acc': float(best_val_acc),
    'total_parameters': total_params
}

with open(f"{OUTPUT_DIR}/bilstm_model_info.json", 'w') as f:
    json.dump(model_info, f, indent=2)

# Save full model
torch.save({
    'model_state_dict': model.state_dict(),
    'class_mapping': class_mapping,
    'idx_to_class': idx_to_class,
    'input_size': INPUT_FEATURE_SIZE,
    'hidden_size': HIDDEN_SIZE,
    'num_layers': NUM_LAYERS,
    'seq_length': SEQ_LENGTH,
    'model_info': model_info
}, f"{OUTPUT_DIR}/full_bilstm_model.pth")

# Export to ONNX
try:
    dummy_input = torch.randn(1, SEQ_LENGTH, INPUT_FEATURE_SIZE).to(device)
    torch.onnx.export(model, dummy_input, f"{OUTPUT_DIR}/bilstm_model.onnx",
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
    print("✓ Exported to ONNX format")
except Exception as e:
    print(f"ONNX export failed (optional): {e}")

# ============================================
# 15. FINAL SUMMARY
# ============================================
print("\n" + "=" * 50)
print("✅ HOÀN THÀNH GIAI ĐOẠN 3.2!")
print("=" * 50)
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"Test accuracy: {accuracy*100:.2f}%")
print(f"Test F1-Score: {f1*100:.2f}%")
print(f"Total model parameters: {total_params:,}")
print(f"Model saved to: {OUTPUT_DIR}")

# Final memory cleanup
torch.cuda.empty_cache()
gc.collect()
print(f"Final GPU memory: {torch.cuda.memory_allocated()/1024**2:.2f} MB")

print("\n📌 Lưu ý: Save version và add /kaggle/working/bilstm_model làm dataset output")